# Chapter 14 - K-Means Clustering

Up to this point every algorithm we have studied has been **supervised**: we handed the
model both features *and* labels, and it learned a mapping from one to the other.
**Unsupervised learning** drops the labels entirely. The goal shifts from prediction to
**discovery** — finding hidden structure, groupings, or compressed representations in
the data itself.

**Clustering** is the most intuitive unsupervised task: given a set of unlabeled data
points, partition them into groups (*clusters*) so that points within a group are more
similar to each other than to points in other groups.

**Topics covered:**
- Unsupervised learning overview
- The K-Means algorithm step by step
- KMeans in scikit-learn
- Visualising K-Means iterations (convergence)
- Choosing K: the elbow method
- Silhouette score and silhouette plots
- K-Means limitations
- K-Means++ initialisation
- Mini-batch K-Means for large datasets
- Practical example: colour quantisation on an image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
sns.set_style('whitegrid')
%matplotlib inline

---
## 1. Unsupervised Learning Overview

In **supervised learning** we have feature-label pairs $(\mathbf{x}_i, y_i)$ and
learn a function $f(\mathbf{x}) \approx y$. In **unsupervised learning** we only
have features $\mathbf{x}_i$ — no target variable at all.

Common unsupervised tasks include:

| Task | Goal | Example algorithms |
|------|------|--------------------|
| **Clustering** | Group similar points | K-Means, DBSCAN, Hierarchical |
| **Dimensionality reduction** | Compress features while preserving structure | PCA, t-SNE |
| **Anomaly detection** | Identify unusual points | Isolation Forest, LOF |
| **Density estimation** | Model the underlying distribution | GMM, KDE |

This notebook focuses on the most popular clustering algorithm: **K-Means**.

---
## 2. Clustering: Grouping Similar Data Points

The intuition is simple: if you look at a scatter plot and see natural "clumps",
clustering formalises the process of assigning each point to one of those clumps.

Let us generate some synthetic data to visualise this.

In [ ]:
# Generate 3-cluster data (we will pretend we don't know the labels)
X_blob, y_true = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: what the algorithm sees (no labels)
axes[0].scatter(X_blob[:, 0], X_blob[:, 1], c='steelblue', edgecolors='k', s=40)
axes[0].set_title('What the algorithm sees — no labels')
axes[0].set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$')

# Right: the hidden ground truth
axes[1].scatter(X_blob[:, 0], X_blob[:, 1], c=y_true, cmap='Set1', edgecolors='k', s=40)
axes[1].set_title('Hidden ground truth — 3 clusters')
axes[1].set_xlabel('$x_1$')
axes[1].set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

---
## 3. The K-Means Algorithm Step by Step

K-Means is an iterative algorithm with a beautifully simple loop:

1. **Initialise** K centroids (e.g. pick K random data points).
2. **Assign** each data point to the nearest centroid.
3. **Update** each centroid to be the mean of the points assigned to it.
4. **Repeat** steps 2--3 until the assignments no longer change (convergence).

The objective being minimised is the **inertia** (also called within-cluster
sum of squares, WCSS):

$$\text{Inertia} = \sum_{i=1}^{n} \| \mathbf{x}_i - \boldsymbol{\mu}_{c(i)} \|^2$$

where $c(i)$ is the cluster assigned to point $i$ and $\boldsymbol{\mu}_{c(i)}$
is that cluster's centroid.

In [ ]:
def kmeans_step_by_step(X, K, max_iter=6, seed=0):
    """Run K-Means manually and record each iteration for visualisation."""
    rng = np.random.RandomState(seed)
    # Step 1: random initialisation
    idx = rng.choice(len(X), size=K, replace=False)
    centroids = X[idx].copy()

    history = []
    for iteration in range(max_iter):
        # Step 2: assign clusters
        distances = np.linalg.norm(X[:, np.newaxis] - centroids[np.newaxis, :], axis=2)
        labels = distances.argmin(axis=1)
        history.append((centroids.copy(), labels.copy()))

        # Step 3: update centroids
        new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])

        # Check convergence
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids

    return history

In [ ]:
history = kmeans_step_by_step(X_blob, K=3, seed=10)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#e74c3c', '#2ecc71', '#3498db']

for i, (centroids, labels) in enumerate(history):
    ax = axes.ravel()[i]
    for k in range(3):
        mask = labels == k
        ax.scatter(X_blob[mask, 0], X_blob[mask, 1], c=colors[k],
                   edgecolors='k', s=30, alpha=0.6)
    ax.scatter(centroids[:, 0], centroids[:, 1], c=colors[:3],
               marker='X', s=250, edgecolors='k', linewidths=2, zorder=5)
    ax.set_title(f'Iteration {i + 1}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

# Hide unused subplots
for j in range(len(history), 6):
    axes.ravel()[j].set_visible(False)

plt.suptitle('K-Means step by step: assign clusters then move centroids',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Notice how the centroids (large **X** markers) move towards the centre of their
assigned points with each iteration, and the cluster assignments stabilise rapidly.

---
## 4. KMeans in Scikit-Learn

In practice we use `sklearn.cluster.KMeans`, which handles initialisation, convergence
checking, and multiple restarts for us.

Key parameters:
- `n_clusters` — the number of clusters K (you must specify this).
- `init` — initialisation method (`'k-means++'` by default).
- `n_init` — number of times the algorithm is run with different seeds; the best
  result (lowest inertia) is kept. Default is 10.
- `max_iter` — maximum iterations per run (default 300).
- `random_state` — for reproducibility.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_blob)

print(f"Cluster labels (first 10):  {kmeans.labels_[:10]}")
print(f"Cluster centres:\n{kmeans.cluster_centers_}")
print(f"Inertia (WCSS):  {kmeans.inertia_:.2f}")
print(f"Iterations:      {kmeans.n_iter_}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_blob[:, 0], X_blob[:, 1], c=kmeans.labels_, cmap='Set1',
           edgecolors='k', s=40)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='gold', marker='X', s=300, edgecolors='k', linewidths=2,
           label='Centroids', zorder=5)
ax.set_title('K-Means (K=3) on make_blobs')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Predicting New Points

Once fit, `kmeans.predict(X_new)` assigns new points to the nearest centroid.
This is useful for assigning incoming data to existing clusters without refitting.

In [ ]:
# Generate some new points and predict their cluster
X_new = np.array([[-5, 0], [0, 5], [5, -5], [0, 0]])
predictions = kmeans.predict(X_new)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_blob[:, 0], X_blob[:, 1], c=kmeans.labels_, cmap='Set1',
           edgecolors='k', s=30, alpha=0.4)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='gold', marker='X', s=300, edgecolors='k', linewidths=2, zorder=5)
ax.scatter(X_new[:, 0], X_new[:, 1], c=predictions, cmap='Set1',
           marker='*', s=400, edgecolors='k', linewidths=2, zorder=6,
           label='New points')
ax.set_title('Predicting cluster membership for new points')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Visualising K-Means Convergence

We can track the inertia at each iteration to see how quickly K-Means converges.
In practice convergence is fast — often within 10--20 iterations.

In [ ]:
# Track inertia per iteration by running with max_iter=1, 2, 3, ...
inertias_per_iter = []
for n_iter in range(1, 21):
    km = KMeans(n_clusters=3, max_iter=n_iter, n_init=1,
                init='random', random_state=42)
    km.fit(X_blob)
    inertias_per_iter.append(km.inertia_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, 21), inertias_per_iter, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Inertia (WCSS)')
ax.set_title('K-Means convergence: inertia decreases with each iteration')
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.show()

---
## 7. Choosing K: The Elbow Method

K-Means requires you to specify the number of clusters **K** up front. But how do
you pick the right K?

The **elbow method** runs K-Means for a range of K values and plots the inertia
(within-cluster sum of squares) against K. As K increases, inertia always decreases
(more clusters = tighter fit). The trick is to look for the **elbow** — the point
where adding another cluster gives diminishing returns.

At the elbow, the marginal benefit of an extra cluster drops sharply.

In [ ]:
K_range = range(1, 11)
inertias = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_blob)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(K_range, inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Elbow at K=3')
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Inertia (WCSS)')
ax.set_title('Elbow Method: Inertia vs K')
ax.set_xticks(range(1, 11))
ax.legend()
plt.tight_layout()
plt.show()

The plot shows a clear "bend" at K=3 — after that, adding more clusters barely
reduces inertia. This matches our ground truth of 3 clusters.

**Caveat:** the elbow is not always obvious, especially with real-world data. Use
it as guidance, not gospel.

---
## 8. Silhouette Score

The **silhouette coefficient** measures how well each point fits in its assigned
cluster compared to the nearest neighbouring cluster. For a single point *i*:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

where:
- $a(i)$ = mean distance from point *i* to all other points in the *same* cluster.
- $b(i)$ = mean distance from point *i* to all points in the *nearest other* cluster.

The score ranges from -1 (badly misclassified) to +1 (perfectly clustered). A
global average close to 1 indicates well-separated clusters.

In [ ]:
sil_scores = []

for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blob)
    score = silhouette_score(X_blob, labels)
    sil_scores.append(score)
    print(f"K={k:2d}  |  Silhouette score: {score:.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(2, 11), sil_scores, 'o-', color='darkorange', linewidth=2, markersize=8)
ax.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Best at K=3')
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score vs K')
ax.set_xticks(range(2, 11))
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Silhouette Plot

The average silhouette score is a single number, but a **silhouette plot** shows the
per-sample silhouette coefficient for each cluster, giving much richer insight. Ideally,
every cluster should have roughly the same width and all bars should extend past the
average line.

In [ ]:
def silhouette_plot(X, labels, ax=None):
    """Draw a silhouette plot for the given clustering."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    n_clusters = len(np.unique(labels))
    sample_silhouette_values = silhouette_samples(X, labels)
    avg_score = silhouette_score(X, labels)

    y_lower = 10
    for k in range(n_clusters):
        cluster_values = sample_silhouette_values[labels == k]
        cluster_values.sort()
        cluster_size = len(cluster_values)
        y_upper = y_lower + cluster_size

        colour = cm.nipy_spectral(float(k) / n_clusters)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_values,
                         facecolor=colour, edgecolor=colour, alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * cluster_size, str(k), fontsize=11)
        y_lower = y_upper + 10

    ax.axvline(x=avg_score, color='red', linestyle='--',
               label=f'Avg: {avg_score:.3f}')
    ax.set_xlabel('Silhouette coefficient')
    ax.set_ylabel('Cluster')
    ax.set_title(f'Silhouette plot (K={n_clusters})')
    ax.set_yticks([])
    ax.legend(loc='best')
    return ax

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, k in zip(axes, [2, 3, 5]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blob)
    silhouette_plot(X_blob, labels, ax=ax)

plt.tight_layout()
plt.show()

At K=3 each cluster has roughly equal width and all silhouette values are high.
At K=2 or K=5 some clusters are much thinner or have negative values — a sign
that points have been assigned to the wrong cluster.

---
## 10. K-Means Limitations

K-Means makes strong assumptions that do not always hold:

1. **Spherical clusters** — K-Means uses Euclidean distance, so it naturally finds
   round, roughly equal-sized clusters. Elongated, ring-shaped, or irregular shapes
   break it.
2. **Equal-sized clusters** — it tends to partition space into roughly equal Voronoi
   cells, even if the true clusters differ in size.
3. **Sensitive to initialisation** — a bad starting position can lead to a poor local
   minimum (mitigated by K-Means++ and `n_init`).
4. **Must specify K** — you need to decide the number of clusters up front.

Let us see these limitations visually.

In [ ]:
from sklearn.datasets import make_moons, make_circles

datasets = {
    'Moons': make_moons(n_samples=300, noise=0.05, random_state=42),
    'Circles': make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42),
    'Anisotropic': None,  # we build this manually
    'Unequal variance': None,
}

# Anisotropic blobs
X_aniso, y_aniso = make_blobs(n_samples=300, centers=3, random_state=170)
transformation = np.array([[0.6, -0.6], [-0.4, 0.8]])
X_aniso = X_aniso @ transformation
datasets['Anisotropic'] = (X_aniso, y_aniso)

# Unequal variance
X_uneq, y_uneq = make_blobs(n_samples=300, cluster_std=[1.0, 2.5, 0.5],
                              centers=3, random_state=42)
datasets['Unequal variance'] = (X_uneq, y_uneq)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

for col, (name, (X_ds, y_ds)) in enumerate(datasets.items()):
    # Top row: true labels
    axes[0, col].scatter(X_ds[:, 0], X_ds[:, 1], c=y_ds, cmap='Set1',
                         edgecolors='k', s=30)
    axes[0, col].set_title(f'{name} — true labels')

    # Bottom row: K-Means labels
    n_c = len(np.unique(y_ds))
    km = KMeans(n_clusters=n_c, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_ds)
    axes[1, col].scatter(X_ds[:, 0], X_ds[:, 1], c=km_labels, cmap='Set1',
                         edgecolors='k', s=30)
    axes[1, col].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                         c='gold', marker='X', s=200, edgecolors='k', linewidths=2)
    axes[1, col].set_title(f'{name} — K-Means')

plt.suptitle('K-Means limitations: struggles with non-spherical and unequal clusters',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

K-Means completely fails on moons and circles because those shapes are not convex
blobs. For such data, density-based methods like DBSCAN (covered in the next
notebook) are far more appropriate.

---
## 11. K-Means++ Initialisation

Standard random initialisation can lead to poor convergence. **K-Means++** (the
default in scikit-learn since version 0.18) is a smarter initialisation strategy:

1. Pick the first centroid uniformly at random from the data.
2. For each subsequent centroid, pick a data point with probability proportional to
   the squared distance from its nearest existing centroid.

This spreads the initial centroids out, dramatically reducing the chance of a bad
local minimum. The result: faster convergence and better final inertia.

In [ ]:
# Compare random init vs k-means++ across multiple seeds
inertias_random = []
inertias_pp = []

for seed in range(50):
    km_rand = KMeans(n_clusters=3, init='random', n_init=1, random_state=seed)
    km_rand.fit(X_blob)
    inertias_random.append(km_rand.inertia_)

    km_pp = KMeans(n_clusters=3, init='k-means++', n_init=1, random_state=seed)
    km_pp.fit(X_blob)
    inertias_pp.append(km_pp.inertia_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([inertias_random, inertias_pp], labels=['Random init', 'K-Means++'])
ax.set_ylabel('Inertia')
ax.set_title('Inertia distribution across 50 random seeds')
plt.tight_layout()
plt.show()

print(f"Random init — mean inertia: {np.mean(inertias_random):.2f}, "
      f"std: {np.std(inertias_random):.2f}")
print(f"K-Means++  — mean inertia: {np.mean(inertias_pp):.2f}, "
      f"std: {np.std(inertias_pp):.2f}")

K-Means++ consistently achieves lower and more stable inertia. This is why it is
the default in scikit-learn.

---
## 12. Mini-Batch K-Means

Standard K-Means uses the entire dataset for every centroid update. For very large
datasets (hundreds of thousands to millions of points) this becomes slow.

**Mini-Batch K-Means** randomly samples a mini-batch of data points at each iteration
and updates the centroids using only that subset. The result is much faster convergence
at the cost of a slightly higher inertia (usually negligible in practice).

In [ ]:
import time

# Generate a larger dataset for timing comparison
X_large, _ = make_blobs(n_samples=100_000, centers=5, random_state=42)

# Standard K-Means
t0 = time.time()
km_std = KMeans(n_clusters=5, random_state=42, n_init=10)
km_std.fit(X_large)
t_std = time.time() - t0

# Mini-Batch K-Means
t0 = time.time()
km_mb = MiniBatchKMeans(n_clusters=5, random_state=42, batch_size=1000, n_init=10)
km_mb.fit(X_large)
t_mb = time.time() - t0

print(f"{'Method':<20} {'Time (s)':>10} {'Inertia':>15}")
print('-' * 47)
print(f"{'KMeans':<20} {t_std:>10.3f} {km_std.inertia_:>15.0f}")
print(f"{'MiniBatchKMeans':<20} {t_mb:>10.3f} {km_mb.inertia_:>15.0f}")
print(f"\nSpeedup: {t_std / t_mb:.1f}x")

Mini-Batch K-Means is typically an order of magnitude faster with nearly identical
inertia — a great choice for large-scale clustering.

---
## 13. Practical Example: Colour Quantisation on an Image

A fun and visual application of K-Means is **colour quantisation**: reducing the
number of distinct colours in an image. Each pixel is a point in 3D RGB space,
and K-Means finds K representative colours (centroids). Every pixel is then
replaced by its nearest centroid colour.

We will use a programmatically generated image so the notebook is self-contained.

In [ ]:
# Create a synthetic colourful image (gradient + shapes)
def make_synthetic_image(size=200):
    """Generate a colourful synthetic image with gradients and shapes."""
    img = np.zeros((size, size, 3), dtype=np.float64)

    # Background gradient
    for i in range(size):
        for j in range(size):
            img[i, j] = [i / size, 0.3, j / size]

    # Add a red circle
    y, x = np.ogrid[:size, :size]
    mask_circle = (x - size // 3) ** 2 + (y - size // 3) ** 2 < (size // 5) ** 2
    img[mask_circle] = [0.9, 0.1, 0.1]

    # Add a blue rectangle
    img[size // 2:size // 2 + size // 4, size // 2:size // 2 + size // 3] = [0.1, 0.2, 0.9]

    # Add a green triangle region
    for i in range(size // 4):
        left = size // 6 + i
        right = size // 6 + size // 4 - i
        if left < right and size // 2 + i < size:
            img[size // 2 + i, left:right] = [0.1, 0.85, 0.2]

    return np.clip(img, 0, 1)

image = make_synthetic_image(200)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(image)
ax.set_title(f'Original image — {len(np.unique(image.reshape(-1, 3), axis=0))} unique colours')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Reshape image to (n_pixels, 3) for clustering
h, w, c = image.shape
pixels = image.reshape(-1, 3)

K_values = [2, 4, 8, 16]

fig, axes = plt.subplots(1, len(K_values) + 1, figsize=(20, 4))

axes[0].imshow(image)
axes[0].set_title('Original')
axes[0].axis('off')

for ax, K in zip(axes[1:], K_values):
    km = KMeans(n_clusters=K, random_state=42, n_init=10)
    labels = km.fit_predict(pixels)
    quantised = km.cluster_centers_[labels].reshape(h, w, c)
    ax.imshow(np.clip(quantised, 0, 1))
    ax.set_title(f'K={K} colours')
    ax.axis('off')

plt.suptitle('Colour quantisation with K-Means', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

With K=2 the image is reduced to just two dominant tones. As K increases, more
detail is preserved. K=16 is nearly indistinguishable from the original for this
synthetic image. In real photographs, K=32 or K=64 typically gives acceptable quality
while dramatically reducing file size.

---
## 14. Summary

| Concept | Key point |
|---------|----------|
| Unsupervised learning | No labels — discover structure in data |
| K-Means algorithm | Iterate: assign to nearest centroid, update centroid |
| Inertia (WCSS) | Objective minimised by K-Means |
| Elbow method | Plot inertia vs K, look for the bend |
| Silhouette score | Measures cluster quality (-1 to +1) |
| K-Means limitations | Assumes spherical clusters, must specify K |
| K-Means++ | Smart initialisation — default in sklearn |
| Mini-Batch K-Means | Faster variant for large datasets |

---

**Next up:** [02 - Hierarchical Clustering and DBSCAN](02_hierarchical_and_dbscan.ipynb)
— clustering methods that handle non-spherical shapes and do not require specifying K
up front.